# Day 12 — Specialization A: NLP
Token classification (Named Entity Recognition) with HuggingFace.
Uses CoNLL-2003 + `distilbert-base-cased`.

Install: `pip install transformers datasets seqeval evaluate accelerate`


In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer, DataCollatorForTokenClassification
import numpy as np, evaluate

ds = load_dataset('conll2003')
label_list = ds['train'].features['ner_tags'].feature.names
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in id2label.items()}
tok = AutoTokenizer.from_pretrained('distilbert-base-cased')


In [ ]:
def tokenize_and_align(examples):
    enc = tok(examples['tokens'], is_split_into_words=True, truncation=True)
    new_labels = []
    for i, labels in enumerate(examples['ner_tags']):
        word_ids = enc.word_ids(batch_index=i)
        prev = None; row = []
        for wid in word_ids:
            if wid is None: row.append(-100)
            elif wid != prev: row.append(labels[wid])
            else: row.append(-100)
            prev = wid
        new_labels.append(row)
    enc['labels'] = new_labels
    return enc

ds_tok = ds.map(tokenize_and_align, batched=True, remove_columns=ds['train'].column_names)


In [ ]:
model = AutoModelForTokenClassification.from_pretrained('distilbert-base-cased', num_labels=len(label_list), id2label=id2label, label2id=label2id)
metric = evaluate.load('seqeval')

def compute_metrics(p):
    preds, labels = p
    preds = preds.argmax(-1)
    true_p, true_l = [], []
    for pr, la in zip(preds, labels):
        tp, tl = [], []
        for pi, li in zip(pr, la):
            if li != -100:
                tp.append(id2label[int(pi)])
                tl.append(id2label[int(li)])
        true_p.append(tp); true_l.append(tl)
    res = metric.compute(predictions=true_p, references=true_l)
    return {'precision': res['overall_precision'], 'recall': res['overall_recall'], 'f1': res['overall_f1']}


In [ ]:
import torch
args = TrainingArguments(
    output_dir='out-ner', per_device_train_batch_size=16, num_train_epochs=2,
    learning_rate=3e-5, eval_strategy='epoch', save_strategy='no',
    fp16=torch.cuda.is_available(), report_to='none', logging_steps=200,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_tok['train'],
                  eval_dataset=ds_tok['validation'], tokenizer=tok,
                  data_collator=DataCollatorForTokenClassification(tok),
                  compute_metrics=compute_metrics)
trainer.train()
print(trainer.evaluate())


In [ ]:
from transformers import pipeline
ner = pipeline('token-classification', model=trainer.model, tokenizer=tok, aggregation_strategy='simple')
ner('Sundar Pichai, the CEO of Google, met Elon Musk in San Francisco yesterday.')
